## DINOv2 LSTM ##

In [ ]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm

from PIL import Image
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix


import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import Dataset, DataLoader


import os
import pickle
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from PIL import Image

## Device

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# device = torch.device("cpu")
device

## Load DinoV2

## Prepare Dataset

In [3]:
# pose_pickle_folder = '/media/osero/SamsungSSD/CMPE_SSD/mmpose-full/0001/User_2_001.pickle'
pose_pickle_folder = '/media/osero/SamsungSSD/CMPE_SSD/mmpose-full/'

def get_active_frames_from_pickle(input_raw) -> np.ndarray:
    threshold = (
        (((input_raw["pose"]["left_hip"][:, 1] + input_raw["pose"]["right_hip"][:, 1]) / 2 )* 7)
        + input_raw["pose"]["nose"][:, 1] * 3
    ) / 10

    active_frames = (
        np.minimum(
            input_raw["hand_left"]["left_lunate_bone"][:, 1],
            input_raw["hand_right"]["right_lunate_bone"][:, 1],
        )
        < threshold
    )

    active_frame_indices = np.argwhere(active_frames).squeeze()
    return active_frame_indices


def get_active_frames(label_name, sample_name):
    pickle_file_name = f"{pose_pickle_folder}/{label_name}/{sample_name}.pickle"
    file = open(pickle_file_name, 'rb')
    input_raw = pickle.load(file)

    return get_active_frames_from_pickle(input_raw)

In [3]:
# Define transform to match the input size for the model
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [5]:
from torch.utils.data import Dataset, DataLoader

frame_frequency = 3

def create_label_dict(classes):
    label_dict = {}
    for i in range(0,len(classes)):
        label_dict[classes[i]] = i
    return label_dict

def crop_list(lst, full_sample_folder):
    if len(lst) > 47:
        start_index = (len(lst) - 47) // 2
        end_index = start_index + 47
        # print('Its cropped:', len(lst))
        # print('full_sample_folder:', full_sample_folder)
        return lst[start_index:end_index]
    return lst

class CustomImageDataset(Dataset):
    def __init__(self, video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-face-c256"):

        full_sample_folder_list = []
        labels = []
        for label_folder in os.listdir(video_folder):
            full_label_folder = os.path.join(video_folder, label_folder)
            label = int(label_folder)
            # print("process_count: ", process_count, ' , label: ', label)
            for sample_folder in os.listdir(full_label_folder):
                full_sample_folder = os.path.join(full_label_folder, sample_folder)
                full_sample_folder_list.append(full_sample_folder)
                labels.append(label)

        self.classes = np.unique(labels)
        label_dict = create_label_dict(self.classes)
        self.labels = [label_dict[x] for x in labels]
        self.full_sample_folder_list = full_sample_folder_list
        self.train_indices = [i for i, string in enumerate(full_sample_folder_list) if not 'user_4' in string.lower()]
        self.test_indices = [i for i, string in enumerate(full_sample_folder_list) if 'user_4' in string.lower()]

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):    
        image_list = []
        full_sample_folder = self.full_sample_folder_list[idx]
        full_sample_folder_files = os.listdir(full_sample_folder)
        splited_paths = self.full_sample_folder_list[idx].split('/')

        active_frame_indices = get_active_frames(splited_paths[-2],splited_paths[-1])
        active_frame_indices = (
            active_frame_indices
            if active_frame_indices.size > 10
            else np.arange(0, len(full_sample_folder_files))
        )
        active_frame_indices = active_frame_indices[0::frame_frequency]
        active_frame_indices = crop_list(active_frame_indices, full_sample_folder)
        full_sample_folder_files = [full_sample_folder_files[i] for i in active_frame_indices]

        for image_file in full_sample_folder_files:
            full_image_file = os.path.join(full_sample_folder, image_file)
            image = Image.open(full_image_file)

            input_tensor = transform(image)  # Add batch dimension
            image_list.append(input_tensor)
            # input_tensor = transform(image).unsqueeze(0).to(device)  # Add batch dimension
            # input_tensor_list.append(input_tensor)

        return image_list, len(image_list), self.labels[idx] 



In [6]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence

# Step 2: Collate function
def collate_fn(batch):
    sequences, lengths, labels = zip(*batch)
    lengths = torch.tensor(lengths)
    labels = torch.tensor(labels)

    # Pad sequences to the maximum length in the batch
    padded_sequences = pad_sequence([torch.stack(seq) for seq in sequences], batch_first=True)

    # Sort by lengths in descending order
    sorted_lengths, sorted_indices = lengths.sort(descending=True)
    sorted_sequences = padded_sequences[sorted_indices]
    sorted_labels = labels[sorted_indices]
    return sorted_sequences, sorted_lengths, sorted_labels

In [7]:
from torch.utils.data import Dataset, DataLoader, Subset

mixed_dataset = CustomImageDataset()

train_dataset = Subset(mixed_dataset, mixed_dataset.train_indices)
test_dataset = Subset(mixed_dataset, mixed_dataset.test_indices)


cc = 5 


In [8]:
batch_size = 1
num_workers = 2

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=num_workers)  # Adjust batch size as needed
4
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=num_workers)

In [ ]:
class_names = mixed_dataset.classes
class_names

input_dim = train_dataset[0][0][0].size(0)  # Get input dimension from a single feature from a video
num_classes = len(set(class_names))
print("input_dim: ", input_dim, " num_classes: ", num_classes)
print("train_dataset size: ", len(train_dataset))
print("test_dataset size: ", len(test_dataset))

## Model

In [ ]:
def split_list(input_list, max_length):
    if max_length <= 0:
        raise ValueError("max_length must be greater than 0")
    return [input_list[i:i + max_length] for i in range(0, len(input_list), max_length)]

class VideoClassifierLSTM(nn.Module):
    def __init__(self, hidden_dim, num_layers, num_classes):
        super(VideoClassifierLSTM, self).__init__()
        self.dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        self.lstm = nn.LSTM(self.dino_model.embed_dim, hidden_dim, num_layers, batch_first=True, bidirectional=False, dropout=0.1)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, lengths):        
        x = x.squeeze()
        # with torch.no_grad():
        feature = self.dino_model(x)
        feature = self.dino_model.norm(feature)
        packed_image_list = pack_padded_sequence(feature.unsqueeze(0), lengths, batch_first=True, enforce_sorted=True)

        _, (hidden, _) = self.lstm(packed_image_list)  # Use last hidden state
        output = self.dropout(hidden[-1])
        output = self.fc(output)  # Take hidden state of the last LSTM layer
        return output
    
hidden_dim = 1024
num_layers = 2
model = VideoClassifierLSTM(hidden_dim=hidden_dim, num_layers=num_layers, num_classes=num_classes)
model = model.to(device)

## Functions

In [11]:
def test_images():
    correct = 0
    top_5_correct = 0
    total = 0
    running_loss = 0.0
    # since we're not training, we don't need to calculate the gradients for our outputs
    test_predicted = []
    test_labels = []

    with torch.no_grad():
        for image_list, lengths, labels in test_loader:
            image_list = image_list.to(device)
            labels = labels.to(device)

            # calculate outputs by running images through the network
            outputs = model(image_list, lengths)
            loss = criterion(outputs, labels)
            
            # the class with the highest energy is what we choose as prediction
            _, predicted = torch.topk(outputs.data, 1)
            _, predicted_top_5 = torch.topk(outputs.data, 5)
            total += labels.size(0)
            correct += (predicted.flatten()== labels.flatten()).sum().item() 
            top_5_correct += sum([(predicted_top_5[i] == labels[i]).any().item() for i in range(len(labels))])
            running_loss += loss.item()

            test_labels += (labels.cpu().numpy().tolist())
            test_predicted += (predicted.cpu().numpy().tolist())

    avg_loss = running_loss / total
    accuracy = 100 * correct / total
    top_5_accuracy = 100 * top_5_correct / total
    print(f'Accuracy of the network on the {len(test_loader)*batch_size} test video: {accuracy:.4f} %, top5: {top_5_accuracy:.4f} %, avg_loss: {avg_loss}')
    return accuracy, top_5_accuracy, avg_loss


In [12]:
import datetime
from time import gmtime, strftime
def get_current_time():
    return strftime("%Y-%m-%d_%H-%M-%S", gmtime())

def save_model_result(current_time):
    result_name = 'lstm_results/FINE_LSTM_RL_BI_' + current_time + '.pth'
    torch.save({'name': result_name,
                'model_state_dict': model.state_dict(),
                'lr': lr,
                'step_size': step_size,
                'gamma': gamma,
                'weight_decay': weight_decay,
                'hidden_dim': hidden_dim,
                'num_layers': num_layers,
                'batch_size': batch_size,
                'frame_frequency': frame_frequency,
                'input_dim': input_dim,
                'num_classes': num_classes,
                'train_dataset': len(train_dataset),
                'test_dataset': len(test_dataset),
                'avg_loss_list': avg_loss_list,
                'avg_accuracy_list': avg_accuracy_list,
                'avg_test_accuracy_list': avg_test_accuracy_list,
                'avg_top5_test_accuracy_list': avg_top5_test_accuracy_list,
                'avg_test_loss_list': avg_test_loss_list},
                result_name)



In [13]:
import shutil
def copy_ipynb_file(current_time): 
    current_file = 'dinofine_tune_lstm_left.ipynb'
    copy_file = '/home/osero/Desktop/CMPE/dinov2/classsification/lstm/lstm_results/ipynbs/COPY_' + current_time + '_' + current_file
    shutil.copy(current_file, copy_file)

## Train

In [ ]:
lr = 1e-4
lr_dino = 1e-6
step_size = 5
gamma = 0.5
weight_decay = 0

criterion = nn.CrossEntropyLoss()
# optimizer = optim.AdamW(model.parameters(), lr=lr)
optimizer = optim.AdamW(
    [
        {"params": model.dino_model.parameters(), "lr": lr_dino},  # Specific LR for dino_model
    ],
    lr=lr,  # Default LR for all other parameters
)
scheduler = lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma) ## CosineAnnealingLR Dene
print(f"lr {lr}, step_size: {step_size}, gamma: {gamma}, weight_decay: {weight_decay}")
print(f"Model hidden_dim {hidden_dim}, num_layers: {num_layers}")
print(f"batch_size {batch_size}, frame_frequency: {frame_frequency}")

avg_loss_list = []
avg_accuracy_list = []
avg_test_accuracy_list = []
avg_top5_test_accuracy_list = []
avg_test_loss_list = []

num_epoch = 10
for epoch in range(num_epoch):
    train_acc = 0
    train_loss = 0
    loop = tqdm(train_loader)

    running_loss = 0.0
    running_accuracy= 0.0
    for idx, (image_list, lengths, labels) in enumerate(loop):
        image_list = image_list.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(image_list, lengths)
        loss = criterion(outputs, labels)

        predictions = outputs.argmax(dim=1, keepdim=True).squeeze()
        correct = (predictions == labels).sum().item()
        accuracy = correct / batch_size

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_accuracy += 100 * accuracy
        loop.set_description(f"Epoch [{epoch}/{num_epoch}]")
        loop.set_postfix(loss=loss.item(), acc=accuracy)
    scheduler.step()
    avg_loss = running_loss / len(train_loader)
    avg_accuracy = running_accuracy / len(train_loader)
    print(f"Time: {get_current_time()} Epoch [{epoch}], Avg loss: {avg_loss:.4f}, Avg accuracy: {avg_accuracy:.4f}")
    avg_test_accuracy, avg_top5_test_accuracy, avg_test_loss = test_images()

    avg_loss_list.append(avg_loss)
    avg_accuracy_list.append(avg_accuracy)
    avg_test_accuracy_list.append(avg_test_accuracy)
    avg_top5_test_accuracy_list.append(avg_top5_test_accuracy)
    avg_test_loss_list.append(avg_test_loss)
    save_model_result('SAVED_FINETUNE_FACE_LSTM')
current_time = get_current_time()
save_model_result(current_time)


In [ ]:
import matplotlib.pyplot as plt
import torch

# summarize history for accuracy
plt.plot(avg_accuracy_list) 
plt.plot(avg_test_accuracy_list)
plt.plot(avg_top5_test_accuracy_list)
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['Train', 'Test', 'Test Top-5'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(avg_loss_list)
plt.plot(avg_test_loss_list)
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

In [ ]:
import torch
import torch.nn as nn

DINO_PATH_FINETUNED_DOWNLOADED="/media/osero/SamsungSSD/yedek files/dinov2-1_files/eval/teacher_checkpoint.pth"

def get_dino_finetuned_downloaded():
    # load the original DINOv2 model with the correct architecture and parameters. The positional embedding is too large.
    # load vits or vitg
    model=torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
    #model=torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14')
    # load finetuned weights
    pretrained = torch.load(DINO_PATH_FINETUNED_DOWNLOADED, map_location=torch.device('cpu'))
    # make correct state dict for loading
    new_state_dict = {}
    for key, value in pretrained['teacher'].items():
        if 'dino_head' in key:
            print('not used')
        else:
            new_key = key.replace('backbone.', '')
            new_state_dict[new_key] = value
    #change shape of pos_embed, shape depending on vits or vitg
    pos_embed = nn.Parameter(torch.zeros(1, 257, 384))
    #pos_embed = nn.Parameter(torch.zeros(1, 257, 1536))
    model.pos_embed = pos_embed
    # load state dict
    model.load_state_dict(new_state_dict, strict=True)
    return model

image = Image.open('/media/osero/SamsungSSD/CMPE_SSD/frame-face-c256/0001/User_2_001/001.jpg')
input_tensor = transform(image).to(device)  # Add batch dimension

default_dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device) 

with torch.no_grad():
    default_dino_model_result = default_dino_model(input_tensor.unsqueeze(0))

default_dino_model.to('cpu')

model = get_dino_finetuned_downloaded()

with torch.no_grad():
    model_result = model(input_tensor.unsqueeze(0))
aaa = 4